# 🧠 RAG Masterclass — Complete Study Notebook
### Based on [anshlambagit/RAG_Tutorial](https://github.com/anshlambagit/RAG_Tutorial) | [▶️ YouTube Masterclass](https://youtu.be/71MW5WeHdz8)

> **Covers all 5 notebooks from the repo, combined into one study guide:**
> - `1.0_RAG_With_Own_Text.ipynb` → Sections 1–5
> - `2.0_RAG_PDF.ipynb` → Section 6
> - `3.0_RAG_Local_Persist.ipynb` → Section 7
> - `4.0_RAG_Add_Doc.ipynb` → Section 8
> - `5_Docker_Model_Runner.ipynb` → Section 9

---

## 🗺️ What Is RAG?

**RAG = Retrieval-Augmented Generation**

LLMs know a lot — but they don't know YOUR data, and they have a knowledge cutoff.
RAG solves this by fetching relevant information from your own documents at query time
and injecting it into the LLM prompt as context.

```
                    ┌─ INDEXING (offline, run once) ────────────────────┐
Your Documents      │                                                    │
(PDF / text / web)  │  Load → Chunk → Embed → Store in Vector DB        │
                    └────────────────────────────────────────────────────┘

                    ┌─ RETRIEVAL (online, per query) ────────────────────┐
User Question  ──→  │  Embed query → Search Vector DB → Top-k chunks     │
                    │  Inject chunks into prompt → LLM → Answer          │
                    └────────────────────────────────────────────────────┘
```

**Why RAG beats fine-tuning for most use cases:**

| | RAG | Fine-tuning |
|---|---|---|
| Update knowledge | Add docs (minutes) | Retrain model (hours–days) |
| Cost | Low (retrieval only) | High (GPU compute) |
| Hallucination risk | Lower — grounded in real docs | Higher — baked-in patterns |
| Best for | Private/dynamic data | Changing model style or behavior |

---

## 📦 Libraries Used in This Repo

| Library | Purpose |
|---|---|
| `langchain_anthropic` | Claude LLM (`ChatAnthropic`) — used for all LLM calls |
| `langchain_huggingface` | `HuggingFaceEmbeddings` — free local embedding model |
| `langchain_community.document_loaders` | `PyPDFLoader` — extracts text from PDF files |
| `langchain_core.documents` | `Document` class — wraps text + metadata |
| `langchain_text_splitters` | `RecursiveCharacterTextSplitter` — smart chunking |
| `langchain_community.vectorstores` | `Chroma` — in-memory and persisted vector store |
| `python-dotenv` | Loads `OPENAI_API_KEY` from `.env` file |

### Install all dependencies
```bash
pip install langchain-anthropic langchain-huggingface langchain-community langchain-text-splitters chromadb pypdf python-dotenv sentence-transformers
```

### `.env` file setup
```
ANTHROPIC_API_KEY=sk-ant-xxxxxxxxxxxx
```

> HuggingFace embeddings run **locally** — no API key needed.


---
## 🔧 Setup — Common to All Notebooks

In [31]:
import os
from dotenv import load_dotenv

# load_dotenv() reads your .env file and pushes secrets into os.environ
# Must run BEFORE initializing any model
load_dotenv()

# .env file should contain:
#   ANTHROPIC_API_KEY=sk-ant-xxxx
# HuggingFace embeddings run locally — no API key needed for them

if os.environ.get('ANTHROPIC_API_KEY'):
    print("✅ Anthropic API Key is set.")
else:
    print("❌ ANTHROPIC_API_KEY not found — check your .env file")


✅ Anthropic API Key is set.


In [32]:
from langchain_anthropic import ChatAnthropic

# ChatAnthropic: LangChain wrapper around Anthropic's Claude API
# model      : claude-haiku-4-5 = fastest & cheapest Claude model
#               Perfect for RAG — most of the "intelligence" comes from
#               the retrieved context, not the model's creativity
# temperature: 0 = deterministic/factual responses (essential for RAG)
#              Higher = more creative but less reliable for grounded answers
model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# Alias as 'llm' too — used throughout the notebook
llm = model
print("✅ Claude Haiku initialized")


✅ Claude Haiku initialized


---
# 📓 Notebook 1 — RAG With Your Own Text (`1.0_RAG_With_Own_Text.ipynb`)

The simplest form of RAG: paste your own text, build a searchable store, query it.

## The 5-Step RAG Process (from the RAG Notes diagram)

```
Step 1: Document Input     →  Wrap your text in a Document object
Step 2: Chunking           →  Split into smaller overlapping pieces
Step 3: Embed + Map        →  Convert chunks to vectors
Step 4: Storage            →  Store vectors in Chroma
Step 5: Semantic Search    →  Find relevant chunks, pass to LLM
```


#### **STEP 1: Preparing Document for your Text**

In [33]:
from langchain_core.documents import Document

# ─── Your raw text ────────────────────────────────────────────────────────────
# This is the knowledge you want to make searchable.
# In production this would come from a database, file, or API.
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform
tasks typically associated with human intelligence, such as learning, reasoning,
problem-solving, perception, and decision-making. It is a field of research in
computer science that develops and studies methods and software that enable machines
to perceive their environment and use learning and intelligence to take actions that
maximize their chances of achieving defined goals.

High-profile applications of AI include advanced web search engines (e.g., Google Search);
recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants
(e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo);
generative and creative tools (e.g., language models and AI art); and superhuman play
and analysis in strategy games (e.g., chess and Go). However, many AI applications are
not perceived as AI: "A lot of cutting edge AI has filtered into general applications,
often without being called AI because once something becomes useful enough and common
enough it's not labeled AI anymore."

Various subfields of AI research are centered around particular goals and the use of
particular tools. The traditional goals of AI research include learning, reasoning,
knowledge representation, planning, natural language processing, perception, and support
for robotics. To reach these goals, AI researchers have adapted and integrated a wide
range of techniques, including search and mathematical optimization, formal logic,
artificial neural networks, and methods based on statistics, operations research, and
economics. AI also draws upon psychology, linguistics, philosophy, neuroscience, and
other fields. Some companies, such as OpenAI, Google DeepMind and Meta, aim to create
artificial general intelligence (AGI) – AI that can complete virtually any cognitive
task at least as well as a human.
"""

# ─── Document object ──────────────────────────────────────────────────────────
# Document: LangChain's core data structure for RAG
#   page_content : the text the LLM will read
#   metadata     : any key-value info you want to track (source, ID, author, date…)
#                  metadata is PRESERVED through chunking — each chunk keeps it
#                  useful for citations, filtering, and debugging

docs = [Document(page_content=my_text, metadata={"source": "ABC", "documentID": "Doc1"})]
docs


[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform\ntasks typically associated with human intelligence, such as learning, reasoning,\nproblem-solving, perception, and decision-making. It is a field of research in\ncomputer science that develops and studies methods and software that enable machines\nto perceive their environment and use learning and intelligence to take actions that\nmaximize their chances of achieving defined goals.\n\nHigh-profile applications of AI include advanced web search engines (e.g., Google Search);\nrecommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants\n(e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo);\ngenerative and creative tools (e.g., language models and AI art); and superhuman play\nand analysis in strategy games (e.g., chess and Go). However, many AI applications are\nnot perceived as AI: "A l

### 💡 Why wrap text in a `Document`?

The `Document` object is the atomic unit in LangChain's RAG pipeline.
Every loader, splitter, vector store, and retriever is designed to work
with `Document` objects — so wrapping text this way makes it compatible
with the entire ecosystem.

The `metadata` dict carries through all downstream steps:
```python
# After chunking, each chunk still has the original metadata:
chunk.metadata  # → {"source": "ABC", "documentID": "Doc1"}
```


#### **STEP 2: Splitting the Document into CHUNKS**

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ─── Why chunk? ───────────────────────────────────────────────────────────────
# 1. LLMs have a context window limit — can't send a 100-page doc
# 2. Smaller chunks = more precise retrieval (only relevant parts are fetched)
# 3. Sending irrelevant content wastes tokens and confuses the model

# ─── RecursiveCharacterTextSplitter ──────────────────────────────────────────
# The recommended general-purpose splitter.
# It tries to split on natural text boundaries in this order:
#   ["\n\n", "\n", " ", ""]
# This keeps paragraphs and sentences intact where possible.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # max characters per chunk
    chunk_overlap=50    # characters repeated between adjacent chunks
                        # WHY overlap? Prevents losing context at boundaries.
                        # A sentence split across two chunks won't lose its start.
)

chunks = splitter.split_documents(docs)
chunks


[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform\ntasks typically associated with human intelligence, such as learning, reasoning,\nproblem-solving, perception, and decision-making. It is a field of research in\ncomputer science that develops and studies methods and software that enable machines\nto perceive their environment and use learning and intelligence to take actions that\nmaximize their chances of achieving defined goals.'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines (e.g., Google Search);\nrecommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants\n(e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo);\ngenerative and creative tools (e.g., language models and AI art); and superhuman play\nand analysis in strategy games (e.g., c

### 💡 `chunk_overlap` visualized

```
chunk_size=500, chunk_overlap=50:

[──────────── chunk 1 (500 chars) ────────────────]
                                     [── overlap ──][─── chunk 2 (500 chars) ───────────────]

The overlap zone (50 chars) appears in BOTH chunks.
This preserves meaning across chunk boundaries.
```

### Choosing `chunk_size` and `chunk_overlap`

| Document type | Recommended chunk_size | chunk_overlap |
|---|---|---|
| Short articles / notes | 300–500 | 30–50 |
| Technical docs / PDFs | 800–1200 | 100–150 |
| Code | 1000–2000 | 100–200 |
| Legal / dense text | 500–800 | 50–100 |

**Rule:** Smaller chunks = more precise retrieval but more fragmented context.
Larger chunks = more context per retrieval but less precise targeting.


#### **STEP 3: Creating Embeddings for the Chunks**

In [35]:
from langchain_huggingface import HuggingFaceEmbeddings

# ─── What is an embedding? ────────────────────────────────────────────────────
# An embedding converts text into a list of numbers (a vector) that captures
# the SEMANTIC MEANING of the text.
#
# Texts with similar meaning produce similar vectors (close in vector space).
# This powers semantic search — "What is AI?" finds chunks about
# "artificial intelligence" even if the exact words don't match.
#
# Example:
#   "What is AI?"                → [0.12, -0.87, 0.43, ...]   (384 numbers)
#   "Define artificial intel."   → [0.11, -0.85, 0.44, ...]   ← very similar!
#   "I love eating pizza"        → [-0.34, 0.12, -0.91, ...]  ← very different

# ─── Why HuggingFace over OpenAI embeddings? ─────────────────────────────────
# ✅ FREE — runs locally on your CPU, no API calls, no cost per embedding
# ✅ PRIVATE — your document text never leaves your machine
# ✅ OFFLINE — works without internet after first download
# ✅ FAST — small model loads quickly, great for experimentation
#
# all-MiniLM-L6-v2:
#   - 384 dimensions (vs 1536 for OpenAI text-embedding-3-small)
#   - ~90MB model download (cached in ~/.cache/huggingface/ after first run)
#   - Excellent English quality for its size
#   - Industry standard for local/free RAG setups
#
# First run downloads the model weights. Subsequent runs load from cache.

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ HuggingFace embedding model loaded")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1221.30it/s]


✅ HuggingFace embedding model loaded


### 💡 How does the vector store find relevant chunks?

It uses **cosine similarity** — measures the angle between two vectors:

```
cos(θ) = (A · B) / (|A| × |B|)

1.0  = identical meaning  (angle = 0°)
0.0  = unrelated          (angle = 90°)
-1.0 = opposite meaning   (angle = 180°)
```

When you run `similarity_search("What is AI?", k=3)`:
1. Your query gets embedded → query vector
2. Query vector is compared to ALL chunk vectors
3. Top-k chunks by cosine similarity are returned

This is why it works even when the query wording doesn't exactly match the chunks.


#### **STEP 4: Create and Store Embeddings in Vector Store**

In [6]:
from langchain_community.vectorstores import Chroma

# ─── Chroma Vector Store (in-memory) ─────────────────────────────────────────
# Chroma is an open-source vector database — great for local development.
# from_documents() does 3 things in one call:
#   1. Embeds every chunk using embedding_model
#   2. Stores the raw text + metadata
#   3. Indexes the vectors for fast similarity search

# Without persist_directory → stored in RAM only (lost when kernel restarts)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# What's happening under the hood:
# chunk 1 text → embedding_model → [0.12, -0.87, ...] → stored in Chroma
# chunk 2 text → embedding_model → [0.34,  0.11, ...] → stored in Chroma
# ...
print(f"✅ Vector store created with {vectorstore._collection.count()} chunks")


✅ Vector store created with 5 chunks


#### **STEP 5: Semantic Search**

In [7]:
# ─── Similarity Search ────────────────────────────────────────────────────────
# similarity_search():
#   1. Embeds the query string
#   2. Finds the k most similar chunks by cosine similarity
#   3. Returns them as Document objects (text + metadata preserved)

results = vectorstore.similarity_search("What is AI?", k=3)
results


[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform\ntasks typically associated with human intelligence, such as learning, reasoning,\nproblem-solving, perception, and decision-making. It is a field of research in\ncomputer science that develops and studies methods and software that enable machines\nto perceive their environment and use learning and intelligence to take actions that\nmaximize their chances of achieving defined goals.'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='artificial neural networks, and methods based on statistics, operations research, and\neconomics. AI also draws upon psychology, linguistics, philosophy, neuroscience, and\nother fields. Some companies, such as OpenAI, Google DeepMind and Meta, aim to create\nartificial general intelligence (AGI) – AI that can complete virtually any cognitive\ntask at least as well as a human.'

#### **Talk to LLM** — Naive RAG (from the notebook)

In [8]:
# ─── Step 1: Retrieve relevant context ───────────────────────────────────────
context = vectorstore.similarity_search("What is AI?", k=3)

# ─── Step 2: Pass context to LLM ─────────────────────────────────────────────
# This is the "Augmented" part of RAG:
# We inject the retrieved chunks into the prompt so the LLM answers
# based on YOUR data, not just its training knowledge.
#
# Note: This naive f-string approach works but has limitations:
#   - No clear separation of context vs question in the prompt
#   - The LLM might ignore context if not instructed clearly
# → See Section 5 for the proper LCEL chain approach
response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)


# What is AI?

Based on the provided context, **Artificial Intelligence (AI)** is:

## Core Definition
The capability of computational systems to perform tasks typically associated with human intelligence, including:
- Learning
- Reasoning
- Problem-solving
- Perception
- Decision-making

## Key Characteristics
AI is a field of computer science that develops methods and software enabling machines to:
1. **Perceive their environment**
2. **Use learning and intelligence** to take actions
3. **Maximize chances of achieving defined goals**

## Interdisciplinary Nature
AI draws from multiple fields including:
- Computer science
- Artificial neural networks
- Statistics and operations research
- Economics
- Psychology, linguistics, philosophy, and neuroscience

## Future Direction
Some leading companies (OpenAI, Google DeepMind, Meta) are working toward **Artificial General Intelligence (AGI)** — AI systems capable of completing virtually any cognitive task at least as well as humans.

## In

### 💡 What just happened? (Full RAG flow recap)

```
Your text (my_text)
    │
    ▼
Document(page_content=..., metadata={...})      ← Step 1
    │
    ▼
RecursiveCharacterTextSplitter → 5 chunks       ← Step 2
    │
    ▼
HuggingFaceEmbeddings → 5 vectors (384-dim each)    ← Step 3
    │
    ▼
Chroma.from_documents() → indexed store         ← Step 4
    │
    ▼
similarity_search("What is AI?", k=3)           ← Step 5
    │
    ▼
Top 3 chunks → injected into prompt → LLM → Answer
```


---
# 📓 Notebook 2 — RAG With PDF (`2.0_RAG_PDF.ipynb`)

The only difference from Notebook 1: **Step 1 uses `PyPDFLoader` instead of manual text**.
Everything else (chunking, embedding, vectorstore, search) is identical.

**PDF used:** `fabric-onelake.pdf` — Microsoft Fabric OneLake documentation


#### **STEP 1: Extracting Text from PDFs**

In [12]:
from langchain_community.document_loaders import PyPDFLoader

# ─── PyPDFLoader ──────────────────────────────────────────────────────────────
# Reads a PDF file and returns one Document per PAGE.
# Each Document gets automatic metadata:
#   {"source": "path/to/file.pdf", "page": 0}   ← page 0 = first page
#   {"source": "path/to/file.pdf", "page": 1}   ← page 1 = second page
#   ...
#
# .load() reads all pages into memory at once.
# For very large PDFs, use .lazy_load() to stream pages one at a time.

loader = PyPDFLoader("../Docs/fabric-onelake.pdf")
docs = loader.load()

print(f"Pages loaded: {len(docs)}")
print(f"Auto-generated metadata: {docs[0].metadata}")
print(f"First 200 chars of page 1:\n{docs[0].page_content[:200]}")


Pages loaded: 382
Auto-generated metadata: {'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creator': 'Microsoft Learn', 'creationdate': '2025-12-12T17:01:10+00:00', 'title': 'fabric onelake | Microsoft Learn', 'moddate': '2025-12-12T17:01:10+00:00', 'source': '../Docs/fabric-onelake.pdf', 'total_pages': 382, 'page': 0, 'page_label': '1'}
First 200 chars of page 1:
Tell us about your PDF experience.
OneLake in Microsoft Fabric
documentation
OneLake is a single, unified, logical data lake for the whole organization. OneLake comes
automatically with every Microsof


### **Creating Own Metadata for PDF Chunks**

In [14]:
# ─── Overriding auto-generated metadata ──────────────────────────────────────
# PyPDFLoader creates metadata automatically (source + page number).
# You can OVERRIDE it to add your own fields.
#
# WHY? The auto-generated metadata tracks pages, but you might want:
#   - A business-friendly source name
#   - Author / developer / version info
#   - Custom tags for filtering
#
# Here Ansh replaces the default metadata with custom fields:
for doc in docs:
    doc.metadata = {
        "source": "fabric-onelake.pdf",
        "developer": "Microsoft"
    }

# Verify the override worked
print("Updated metadata:", docs[0].metadata)


Updated metadata: {'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}


### 💡 Should you override or extend metadata?

**Override** : replaces all metadata with your own fields.
Simpler but loses the auto-generated page numbers.

**Extend** (recommended for production): keep page numbers + add your own:
```python
for doc in docs:
    doc.metadata["developer"] = "Microsoft"     # add new field
    doc.metadata["source"] = "fabric-onelake"   # override just one field
    # doc.metadata["page"] is still preserved!
```
Page numbers are valuable — they let you cite "Answer found on page 12."


#### **STEP 2: Splitting the Document into CHUNKS**

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ─── Larger chunks for PDF technical docs ─────────────────────────────────────
# PDFs often have denser content — 1000 char chunks capture more context
# chunk_overlap=100 gives more boundary protection than the 50 used for short text
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

# Verify metadata persists through chunking
print(f"Chunk metadata: {chunks[0].metadata}")


Total chunks: 652
Chunk metadata: {'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}


In [16]:
# Inspect a specific chunk's metadata
# This confirms: metadata is inherited by ALL chunks from the parent document
chunks[0].metadata


{'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}

#### **STEP 3: Creating Embeddings for the Chunks**

In [36]:
from langchain_huggingface import HuggingFaceEmbeddings

# Same embedding model as Notebook 1 — the model doesn't care whether
# the text came from a PDF or a hardcoded string. It just embeds text.
#
# CRITICAL RULE: You MUST use the same embedding model at index time AND query time.
# If you build the store with all-MiniLM-L6-v2 and query with a different model,
# the vectors are incompatible — results will be nonsense.
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embedding model ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2530.66it/s]


✅ Embedding model ready


#### **STEP 4: Create and Store Embeddings in Vector Store**

In [19]:
from langchain_community.vectorstores import Chroma

# In-memory vector store (same as Notebook 1, no persist_directory)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print(f"✅ Indexed {vectorstore._collection.count()} chunks from PDF")


✅ Indexed 657 chunks from PDF


#### **STEP 5: Semantic Search**

In [37]:
# Query the PDF content with a natural language question
# The retriever finds the most relevant sections even if the exact words differ
results = vectorstore.similarity_search("Why do we need OneLake?", k=3)

for i, doc in enumerate(results):
    print(f"─── Result {i+1} ───")
    print(f"Source: {doc.metadata}")
    print(f"Content: {doc.page_content[:250]}...")
    print()


─── Result 1 ───
Source: {'developer': 'Microsoft', 'source': 'fabric-onelake.pdf'}
Content: OneLake accepts almost all of the same headers as Storage, ignoring only some headers
that relate to unpermitted actions on OneLake. Since these headers don't alter the
behavior of the entire call, OneLake ignores the banned headers, returns them in ...

─── Result 2 ───
Source: {'developer': 'Microsoft', 'source': 'fabric-onelake.pdf'}
Content: OneLake, the OneDrive for data
Article• 07/25/2024
OneLake is a single, unified, logical data lake for your whole organization. A data Lake
processes large volumes of data from various sources. Like OneDrive, OneLake comes
automatically with every Mi...

─── Result 3 ───
Source: {'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}
Content: If a tool or package compatible with ADLS isn't working over OneLake, the most common issue
is URL validation. As OneLake uses a different endpoint (dfs.fabric.microsoft.com) than ADLS
(dfs.core.windows.net), so

#### **Talk to LLM**

In [21]:
context = vectorstore.similarity_search("What is AI?", k=3)

response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)


# What is AI?

Based on the provided context, **Artificial Intelligence (AI)** is:

## Core Definition
The capability of computational systems to perform tasks typically associated with human intelligence, including:
- Learning
- Reasoning
- Problem-solving
- Perception
- Decision-making

## Key Characteristics
AI is a field of computer science that develops methods and software enabling machines to:
1. **Perceive their environment** through various sensors and data inputs
2. **Use learning and intelligence** to take actions
3. **Maximize chances of achieving defined goals**

## Interdisciplinary Nature
AI draws from multiple fields including:
- Statistics and operations research
- Economics
- Psychology
- Linguistics
- Philosophy
- Neuroscience

## Current Goals
Some leading companies like OpenAI, Google DeepMind, and Meta are working toward **Artificial General Intelligence (AGI)** — AI systems that can complete virtually any cognitive task at least as well as humans.

## Interesting

---
# 📓 Notebook 3 — Local Persisted Vector Store (`3.0_RAG_Local_Persist.ipynb`)

**The problem with in-memory Chroma:**
Every notebook restart re-embeds ALL documents from scratch.
For a 100-page PDF, this can take minutes and costs OpenAI API credits each time.

**The solution:** Persist the vector store to disk.
Build it once. Load it instantly on every subsequent run.

This is the **production-ready pattern** for RAG.


#### **STEP 1: Extracting Text from PDFs** *(same as Notebook 2)*

In [23]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../Docs/fabric-onelake.pdf")
docs = loader.load()

# Override with custom metadata
for doc in docs:
    doc.metadata = {"source": "fabric-onelake.pdf", "developer": "Microsoft"}

print(f"Loaded {len(docs)} pages")


Loaded 382 pages


### **Creating own Metadata for PDF Chunks** *(same as Notebook 2)*

In [24]:
# Already done above — metadata set to {"source": "fabric-onelake.pdf", "developer": "Microsoft"}
print("Metadata:", docs[0].metadata)


Metadata: {'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}


#### **STEP 2: Splitting the Document into CHUNKS** *(same as Notebook 2)*

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Chunks: {len(chunks)}")
print(f"Metadata on chunk: {chunks[0].metadata}")


Chunks: 652
Metadata on chunk: {'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}


#### **STEP 3: Creating Embeddings for the Chunks** *(same as Notebook 2)*

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embedding model ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3994.39it/s]


✅ Embedding model ready


#### **STEP 4: Create and Store Embeddings in Local Vector Store** ← KEY CHANGE

In [27]:
from langchain_community.vectorstores import Chroma

# ─── persist_directory: THE KEY DIFFERENCE from in-memory ────────────────────
# Adding persist_directory tells Chroma to SAVE everything to disk.
#
# Chroma creates this folder structure:
#   ./Vector/
#   ├── chroma.sqlite3          ← text content, metadata, collection info
#   └── <uuid>/
#       ├── data_level0.bin     ← HNSW vector index (the search structure)
#       ├── header.bin
#       ├── length.bin
#       └── link_lists.bin
#
# HNSW = Hierarchical Navigable Small World graph
# Enables O(log n) approximate nearest-neighbor search
# Much faster than brute-force O(n) comparison at scale

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector/"   # ← save to this folder
)

print(f"✅ Vector store PERSISTED to './Vector/'")
print(f"   Chunks stored: {vectorstore._collection.count()}")


✅ Vector store PERSISTED to './Vector/'
   Chunks stored: 652


#### **STEP 5: Semantic Search** *(same as before)*

In [28]:
results = vectorstore.similarity_search("Why do we need OneLake?", k=3)
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content[:200]}...")
    print()


Result 1: OneLake accepts almost all of the same headers as Storage, ignoring only some headers
that relate to unpermitted actions on OneLake. Since these headers don't alter the
behavior of the entire call, On...

Result 2: OneLake, the OneDrive for data
Article• 07/25/2024
OneLake is a single, unified, logical data lake for your whole organization. A data Lake
processes large volumes of data from various sources. Like O...

Result 3: If a tool or package compatible with ADLS isn't working over OneLake, the most common issue
is URL validation. As OneLake uses a different endpoint (dfs.fabric.microsoft.com) than ADLS
(dfs.core.windo...



#### **Re-Use the Vector Database** ← THE PAYOFF

In [29]:
# ─── Load from disk (subsequent runs) ────────────────────────────────────────
# Instead of Chroma.from_documents() (which re-embeds everything),
# use Chroma() directly to LOAD the existing persisted store.
#
# This is INSTANT — no API calls, no re-embedding.
# The vectors are already computed and saved on disk.
#
# IMPORTANT: You must pass the SAME embedding_function that was used when building.
# Chroma doesn't store the embedding model — only the vectors it produced.
# Using a different model would produce incompatible vectors.

vectorstore_persist = Chroma(
    persist_directory="./Vector/",         # where the store was saved
    embedding_function=embedding_model     # same model used to build it!
)

print(f"✅ Loaded from disk — {vectorstore_persist._collection.count()} chunks ready")


✅ Loaded from disk — 652 chunks ready


C:\Users\baksh\AppData\Local\Temp\ipykernel_17368\598823322.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(


In [30]:
# Works exactly like the original — no difference from the user's perspective
vectorstore_persist.similarity_search("Why do we need OneLake?", k=3)


[Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content="OneLake accepts almost all of the same headers as Storage, ignoring only some headers\nthat relate to unpermitted actions on OneLake. Since these headers don't alter the\nbehavior of the entire call, OneLake ignores the banned headers, returns them in a new\n'x-ms-rejected-headers' response header, and permits the rest of the call. For example,\nManaged OneLake folders\nUnsupported request headers and parameters"),
 Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content='OneLake, the OneDrive for data\nArticle• 07/25/2024\nOneLake is a single, unified, logical data lake for your whole organization. A data Lake\nprocesses large volumes of data from various sources. Like OneDrive, OneLake comes\nautomatically with every Microsoft Fabric tenant and is designed to be the single place\nfor all your analytics data. OneLake brings customers:\nOne data lake for the e

### 💡 In-memory vs. Persisted — When to use each

| | In-Memory | Persisted |
|---|---|---|
| `from_documents()` | Always re-embeds | Build once |
| Survives restart | ❌ No | ✅ Yes |
| Speed after first build | N/A | Instant load |
| Best for | Prototyping / testing | Production / large docs |

### 💡 Smart Loader Pattern (production best practice)

```python
import os

def get_vectorstore(docs=None, persist_dir="./Vector/"):
    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        # Already exists on disk — load it (no API calls)
        return Chroma(persist_directory=persist_dir, embedding_function=embedding_model)
    else:
        # First run — build and save
        return Chroma.from_documents(docs, embedding_model, persist_directory=persist_dir)
```


---
# 📓 Notebook 4 — Add Documents to Existing Store (`4.0_RAG_Add_Doc.ipynb`)

**Scenario:** You already built a vector store from `fabric-onelake.pdf` (Notebook 3).
Now you have a NEW document: `fabric-admin.pdf`.

**Goal:** Add it to the EXISTING persisted store — without rebuilding from scratch.

This is essential for real systems where your knowledge base grows over time.


#### **STEP 1: Extracting Text from PDFs** — New document this time

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# NEW document — fabric-admin.pdf instead of fabric-onelake.pdf
loader = PyPDFLoader("./Docs/fabric-admin.pdf")
docs = loader.load()
print(f"New document pages: {len(docs)}")


### **Creating own Metadata for PDF Chunks**

In [ ]:
# Give the new document its own metadata
# This is how you distinguish between documents in the same vector store
for doc in docs:
    doc.metadata = {
        "source": "fabric-admin.pdf",    # different source from Notebook 3!
        "developer": "Microsoft"
    }

print("Metadata:", docs[0].metadata)


#### **STEP 2: Splitting the Document into CHUNKS**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"New chunks: {len(chunks)}")
print(f"Metadata: {chunks[0].metadata}")


#### **STEP 3: Creating Embeddings for the Chunks**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# MUST be the same model used to build the existing store in Notebook 3!
# The persisted Chroma store contains 384-dim vectors (all-MiniLM-L6-v2).
# Loading with a different model would produce incompatible query vectors.
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embedding model ready (same as build time)")


#### **STEP 4: Store Embeddings in the EXISTING Local Vector Store** ← KEY CHANGE

In [ ]:
from langchain_community.vectorstores import Chroma

# ─── LOAD the existing store (don't rebuild it!) ──────────────────────────────
# Chroma(): connects to the existing persisted store on disk.
# This is NOT from_documents() — we're not rebuilding, just connecting.
# The existing fabric-onelake.pdf vectors are already in there.

vectorstore = Chroma(
    persist_directory="./Vector/",         # same path as Notebook 3
    embedding_function=embedding_model     # same model as Notebook 3
)

print(f"Existing chunks in store: {vectorstore._collection.count()}")


In [ ]:
# ─── Add the new document's chunks ───────────────────────────────────────────
# .add_documents() embeds and indexes the new chunks
# WITHOUT touching the existing vectors — they stay as-is
#
# After this call, the vector store contains BOTH:
#   - All fabric-onelake.pdf chunks (from Notebook 3)
#   - All fabric-admin.pdf chunks (added now)

vectorstore.add_documents(chunks)

print(f"✅ Total chunks after adding fabric-admin.pdf: {vectorstore._collection.count()}")


#### **STEP 5: Semantic Search** — Now searches BOTH documents

In [ ]:
# This query will find relevant chunks from EITHER document
results = vectorstore.similarity_search("What is Microsoft Fabric admin", k=3)

for i, doc in enumerate(results):
    print(f"Result {i+1} | Source: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content[:200]}...")
    print()


#### **Re-Use the Vector Database** — Both docs still available

In [ ]:
# Load again from disk — both documents' vectors are persisted
vectorstore_persist = Chroma(
    persist_directory="./Vector/",
    embedding_function=embedding_model
)

print(f"Loaded {vectorstore_persist._collection.count()} chunks (both docs)")

# OneLake query — finds chunks from fabric-onelake.pdf
results_onelake = vectorstore_persist.similarity_search("Why do we need OneLake?", k=3)
print(f"OneLake results: {len(results_onelake)} chunks")
for r in results_onelake:
    print(f"  Source: {r.metadata.get('source')}")


### 💡 Avoiding Duplicate Documents

If you re-run `add_documents()` on an already-indexed document, you'll get duplicates.
Chroma doesn't auto-deduplicate. Solutions:

**Option 1: Check before adding**
```python
existing_sources = set(
    m["source"] for m in vectorstore.get()["metadatas"]
)
new_chunks = [c for c in chunks if c.metadata["source"] not in existing_sources]
if new_chunks:
    vectorstore.add_documents(new_chunks)
    print(f"Added {len(new_chunks)} new chunks")
else:
    print("Already indexed — skipping")
```

**Option 2: Provide explicit IDs**
```python
# Chroma deduplicates by ID — same ID = overwrite instead of duplicate
ids = [f"fabric-admin-chunk-{i}" for i in range(len(chunks))]
vectorstore.add_documents(chunks, ids=ids)
```


---
# 📓 Notebook 5 — Docker Model Runner (`5_Docker_Model_Runner.ipynb`)

**Run a local LLM — no API key, no cloud, no cost per token.**

Docker Desktop's **Model Runner** lets you run LLMs locally on your machine.
The local model exposes an OpenAI-compatible API at `http://localhost:12434`,
so LangChain's `ChatOpenAI` works without ANY code changes — just swap the `base_url`.

### Setup (one-time)
1. Install [Docker Desktop](https://www.docker.com/products/docker-desktop/)
2. Enable **Model Runner** in Docker Desktop settings
3. Pull a model: `docker model pull ai/qwen3:0.6B-F16`
4. Run: `docker model run ai/qwen3:0.6B-F16`

The model is now available at `http://localhost:12434/engines/llama.cpp/v1`

### Why Docker Model Runner for RAG?
- **No API cost** — run thousands of queries for free
- **Privacy** — your documents never leave your machine
- **Offline** — works without internet
- **Same code** — `ChatOpenAI` works with a different `base_url`


In [ ]:
# ─── Docker Model Runner: an alternative to Claude for the LLM step ──────────
# Everything in this notebook uses Claude (ChatAnthropic) as the LLM.
# Docker Model Runner is shown here as an alternative: a FREE LOCAL LLM
# with zero API cost and full privacy — useful when you want to test
# the RAG pipeline without spending API credits.
#
# It uses ChatOpenAI (not ChatAnthropic) but points base_url to localhost
# instead of api.openai.com — the local Docker model speaks the same API protocol.

from langchain_openai import ChatOpenAI

# base_url: local Docker endpoint (not the cloud)
# model   : "ai/qwen3:0.6B-F16" — a tiny but capable open-source LLM
#   Qwen3 = Alibaba's open-source LLM | 0.6B = 600M params | F16 = 16-bit precision
# No API key needed — Docker ignores it (but LangChain still requires the param)

llm_local = ChatOpenAI(
    base_url="http://localhost:12434/engines/llama.cpp/v1",
    model="ai/qwen3:0.6B-F16",
    temperature=0
)


In [ ]:
# ─── Test the local model ─────────────────────────────────────────────────────
# Works exactly like the cloud version — same .invoke() interface
llm_local.invoke("What is AI?")


In [ ]:
# ─── Print just the text content ─────────────────────────────────────────────
# The response is an AIMessage object — .content gives the text string
response = llm_local.invoke("What is AI?")
print(response.content)


### 💡 Swapping Local ↔ Cloud — Zero Code Change

Because `ChatOpenAI` is just an HTTP client, switching models is one line:

```python
# Cloud (OpenAI)
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# Local (Docker) — same interface, different endpoint + model
llm = ChatOpenAI(
    base_url="http://localhost:12434/engines/llama.cpp/v1",
    model="ai/qwen3:0.6B-F16",
    temperature=0
)

# Your chain code is IDENTICAL — no other changes needed
chain = prompt | llm | StrOutputParser()
chain.invoke({"question": "What is AI?"})
```

### 💡 Local Model Options (Docker Model Runner)

| Model | Size | RAM needed | Best for |
|---|---|---|---|
| `ai/qwen3:0.6B-F16` | 0.6B params | ~2GB | Learning, testing |
| `ai/qwen3:1.7B` | 1.7B params | ~4GB | Better quality |
| `ai/llama3.2:3B` | 3B params | ~6GB | Good balance |
| `ai/mistral:7B` | 7B params | ~8GB | Near-cloud quality |

```bash
# List available models
docker model ls

# Pull a model
docker model pull ai/qwen3:0.6B-F16

# Check it's running
docker model run ai/qwen3:0.6B-F16 "Hello"
```


---
# 🚀 Bonus — Production RAG Chain with LCEL

The notebooks use a naive f-string approach to pass context to the LLM.
This is great for learning but the production pattern uses **LCEL** (LangChain
Expression Language) for a cleaner, composable, reusable chain.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic

# ─── Load embedding model + persisted vector store ───────────────────────────
# MUST use the same embedding model that was used to build the store
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./Vector/", embedding_function=embedding_model)

# ─── Step 1: Retriever ────────────────────────────────────────────────────────
# as_retriever() wraps the vector store in LangChain's standard Retriever interface
# Makes it composable with | pipe syntax
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ─── Step 2: RAG Prompt Template ─────────────────────────────────────────────
# A well-structured RAG prompt:
#   1. Tells the model to use ONLY the provided context (prevents hallucination)
#   2. Tells it to admit when it doesn't know (prevents making things up)
#   3. Keeps {context} and {question} as clearly separated variables
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the question using ONLY
the provided context. If the answer is not in the context, say:
"I cannot find this information in the provided documents."""),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# ─── Step 3: Format retrieved docs into a string ──────────────────────────────
def format_docs(docs):
    """Join Document objects into a single context string for the prompt."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# ─── Step 4: Assemble the full chain ──────────────────────────────────────────
# Data flow when you call chain.invoke("Why do we need OneLake?"):
#
# "Why do we need OneLake?"
#   │
#   ├─ "context" branch:
#   │    retriever.invoke(question) → [Doc1, Doc2, Doc3]
#   │    format_docs([Doc1, Doc2, Doc3]) → "chunk1\n\n---\n\nchunk2..."
#   │
#   └─ "question" branch:
#        RunnablePassthrough() → "Why do we need OneLake?" (unchanged)
#   │
#   ▼
#   rag_prompt → formatted ChatPromptValue
#   │
#   ▼
#   claude-haiku-4-5 → AIMessage  (grounded in retrieved context)
#   │
#   ▼
#   StrOutputParser() → plain string

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

rag_chain = (
    {
        "context": retriever | format_docs,   # retrieve → format → context string
        "question": RunnablePassthrough()     # pass question through unchanged
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# One clean call — the entire pipeline runs end-to-end
answer = rag_chain.invoke("Why do we need OneLake?")
print(answer)


---
# 🗺️ Full RAG Concept Map — Quick Reference

## Architecture (from the RAG Notes diagram)

```
INDEXING (offline)                          RETRIEVAL (real-time)
══════════════════                          ══════════════════════

Documents                                   User Question
    │                                           │
    ▼                                           ▼
PyPDFLoader / Document()               Embed with same model
    │                                           │
    ▼                                           ▼
RecursiveCharacterTextSplitter         Vector similarity search
(chunk_size, chunk_overlap)            (cosine similarity, k=3)
    │                                           │
    ▼                                           ▼
HuggingFaceEmbeddings                       Top-k chunks (Documents)
text → [0.12, -0.87, ...]  (384-dim, free local)                      │
    │                                           ▼
    ▼                                   Inject into prompt template
Chroma.from_documents()                        │
  (in-memory OR persist_directory)             ▼
    │                                   LLM generates grounded answer
    ▼                                           │
Vectors stored + indexed (HNSW)                ▼
                                       User gets answer
```

## Notebook Summary

| Notebook | New Concept | Key Code |
|---|---|---|
| 1.0 — Own Text | Baseline RAG | `Document()`, `Chroma.from_documents()` |
| 2.0 — PDF | PDF loading + custom metadata | `PyPDFLoader`, `doc.metadata = {...}` |
| 3.0 — Persist | Save/load vector store | `persist_directory="./Vector/"`, `Chroma(persist_directory=...)` |
| 4.0 — Add Doc | Update existing store | `vectorstore.add_documents(chunks)` |
| 5.0 — Docker | Free local LLM | `base_url="http://localhost:12434/..."` |

## Key Parameter Reference

| Parameter | Location | Effect |
|---|---|---|
| `chunk_size` | Splitter | Characters per chunk — larger = more context, less precise |
| `chunk_overlap` | Splitter | Characters repeated between chunks — prevents boundary loss |
| `k` | similarity_search | Number of chunks to retrieve — more = richer context, higher cost |
| `temperature` | ChatOpenAI | 0 = deterministic (best for RAG), 1 = creative |
| `persist_directory` | Chroma | Path to save — omit for in-memory, set for production |
| `model` | HuggingFaceEmbeddings | MUST be the same at index time and query time |

## Common Mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Different embedding model at query time | Wrong/irrelevant results | Always use the same model for build and load |
| No `chunk_overlap` | Answers cut off mid-sentence | Set `chunk_overlap=50-150` |
| `k` too low | LLM says "I don't know" (context missed) | Increase k or reduce chunk_size |
| Re-running `from_documents()` on persisted dir | Duplicates | Use `Chroma()` to load, `add_documents()` to extend |
| Hardcoding API key | Security risk | Always use `.env` + `load_dotenv()` |
